# 🔧 題目 1：零售 POS 銷售分析 — Solution

⚠️ **講師用完整答案。學員請用 `pipeline_starter.ipynb`。**

> Pipeline：`CSV → pandas → SQLite (raw/cleaned/analyzed) → SQL → LLM → FastAPI → Streamlit`


## Section 0：環境設定


In [ ]:
import pandas as pd
import sqlite3
import os
import json
print("✅ 套件載入完成")


In [ ]:
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅ API Key" if OPENAI_API_KEY else "⚠️ fallback 模式")


---
## Section 1：Extract


### Step 1-1：讀取 CSV


In [ ]:
df_raw = pd.read_csv("data/raw/topic_1/orders.csv")
print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
print(f"欄位: {list(df_raw.columns)}")
df_raw.head()


### Step 1-2：檢查資料品質


In [ ]:
print("=== 型別 ===")
print(df_raw.dtypes)
print("\n=== 缺漏值 ===")
print(df_raw.isnull().sum())
print("\n=== 數值統計 ===")
print(df_raw.describe())


### Step 1-3：自由探索


In [ ]:
print("=== 國家分佈 ===")
print(df_raw["country"].value_counts().head(10))
print(f"\n商品種類: {df_raw['description'].nunique()}")
print(f"客戶數: {df_raw['customer_id'].nunique()}")


### Step 1-4：SQLite + raw 表


In [ ]:
DB_PATH = "pipeline.db"
conn = sqlite3.connect(DB_PATH)
df_raw.to_sql("raw_orders", conn, if_exists="replace", index=False)
result = pd.read_sql("SELECT COUNT(*) as total FROM raw_orders", conn)
print(f"✅ raw_orders: {result['total'][0]} 筆")


---
## Section 2：Transform


### Step 2-1：從 raw 表讀出


In [ ]:
df = pd.read_sql("SELECT * FROM raw_orders", conn)
before = len(df)
print(f"從 raw_orders 讀出 {before} 筆")


### Step 2-2：處理缺漏值


In [ ]:
df = df.dropna(subset=["description", "customer_id"])
print(f"清洗前: {before} → 清洗後: {len(df)} 筆（刪除 {before - len(df)} 筆）")


### Step 2-3：日期轉換 + 時間特徵


In [ ]:
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["year"] = df["invoice_date"].dt.year
df["month"] = df["invoice_date"].dt.month
df["day_of_week"] = df["invoice_date"].dt.day_name()
df["hour"] = df["invoice_date"].dt.hour
print(df[["invoice_date", "year", "month", "day_of_week", "hour"]].head(3))


### Step 2-4：總金額 + 過濾


In [ ]:
df["total_amount"] = df["quantity"] * df["unit_price"]
df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)]
print(f"最終: {len(df)} 筆")
print(f"總金額範圍: {df['total_amount'].min():.2f} - {df['total_amount'].max():.2f}")


### 🏁 檢查點


In [ ]:
assert df.isnull().sum().sum() == 0
assert (df["quantity"] > 0).all()
assert (df["unit_price"] > 0).all()
assert "year" in df.columns
assert "month" in df.columns
print("✅ 全部檢查通過")
print(f"   {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5：寫入 cleaned 表


In [ ]:
df.to_sql("cleaned_orders", conn, if_exists="replace", index=False)
for t in ["raw_orders", "cleaned_orders"]:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)["n"][0]
    print(f"  {t}: {n} 筆")


---
## Section 3：SQL 統計


### Step 3-1：商品銷售排行


In [ ]:
product_stats = pd.read_sql("""
    SELECT description,
           COUNT(*) as order_count,
           SUM(quantity) as total_qty,
           ROUND(SUM(total_amount), 2) as total_revenue
    FROM cleaned_orders
    GROUP BY description
    ORDER BY total_revenue DESC
    LIMIT 20
""", conn)
product_stats


### Step 3-2：各國銷售統計


In [ ]:
country_stats = pd.read_sql("""
    SELECT country,
           COUNT(DISTINCT customer_id) as customers,
           COUNT(*) as orders,
           ROUND(SUM(total_amount), 2) as revenue
    FROM cleaned_orders
    GROUP BY country
    ORDER BY revenue DESC
""", conn)
country_stats


### Step 3-3：視覺化


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
product_stats.head(10).plot.barh(x="description", y="total_revenue", ax=axes[0], color="steelblue")
axes[0].set_title("商品銷售額 Top 10")
country_stats.head(10).plot.barh(x="country", y="revenue", ax=axes[1], color="coral")
axes[1].set_title("各國銷售額 Top 10")
plt.tight_layout()
plt.show()


### Step 3-4：自由探索


In [ ]:
hourly = pd.read_sql("""
    SELECT hour, COUNT(*) as orders, ROUND(SUM(total_amount), 2) as revenue
    FROM cleaned_orders
    GROUP BY hour
    ORDER BY hour
""", conn)
print("📊 每小時銷售：")
hourly


### Step 3-5：存統計結果


In [ ]:
os.makedirs("processed", exist_ok=True)
product_stats.to_csv("processed/product_stats.csv", index=False)
country_stats.to_csv("processed/country_stats.csv", index=False)
print("✅ 已存到 data/processed/")


---
## Section 4：LLM 加值分析


In [ ]:
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下零售商品描述，回傳 JSON：
{{"category": "家飾/禮品/餐具/季節商品/文具/其他", "insight": "一句話商品洞察"}}
商品描述：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["christmas","xmas","santa","winter"]): cat = "季節商品"
    elif any(w in t for w in ["candle","holder","frame","lamp"]): cat = "家飾"
    elif any(w in t for w in ["cup","mug","plate","bowl"]): cat = "餐具"
    elif any(w in t for w in ["pen","pencil","notebook","card"]): cat = "文具"
    elif any(w in t for w in ["gift","bag","box","ribbon"]): cat = "禮品"
    else: cat = "其他"
    return {"category": cat, "insight": text[:50] + "..."}
print("✅ LLM Helper 已定義")


### Step 4-1：單筆測試


In [ ]:
test = df["description"].iloc[0]
print(f"📝 輸入: {test}")
result = llm_analyze(test, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"🤖 分類: {result['category']}, 洞察: {result['insight']}")


### Step 4-2：批次分析


In [ ]:
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None
print(f"模式: {'API' if api_key else 'Fallback'}, 分析 {BATCH_SIZE} 筆...")

results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(str(row["description"]), api_key)
    results.append(r)
    if len(results) % 10 == 0:
        print(f"  進度: {len(results)}/{BATCH_SIZE}")

print(f"\n✅ 完成 {len(results)} 筆")


### Step 4-3：整理結果 + 寫入 analyzed 表


In [ ]:
df_analyzed = df.head(BATCH_SIZE).copy()
df_analyzed["category"] = [r["category"] for r in results]
df_analyzed["llm_insight"] = [r["insight"] for r in results]

print(f"📊 品類分佈：\n{df_analyzed['category'].value_counts()}")

df_analyzed.to_sql("analyzed_orders", conn, if_exists="replace", index=False)
print(f"\n📊 三表狀態：")
for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)["n"][0]
    print(f"  {t}: {n} 筆")


---
## Section 5：驗證 pipeline


### Step 5-1：跨表查詢


In [ ]:
lineage = pd.read_sql("""
    SELECT 'raw_orders' as layer, COUNT(*) as rows FROM raw_orders
    UNION ALL
    SELECT 'cleaned_orders', COUNT(*) FROM cleaned_orders
    UNION ALL
    SELECT 'analyzed_orders', COUNT(*) FROM analyzed_orders
""", conn)
print("📊 Pipeline 資料流：")
print(lineage.to_string(index=False))


---
## Section 6：產出報告


### Step 6-1：寫報告


In [ ]:
total_rev = pd.read_sql("SELECT ROUND(SUM(total_amount),2) as r FROM cleaned_orders", conn)["r"][0]
top_products = product_stats.head(5)
top_countries = country_stats.head(5)
cat_dist = df_analyzed["category"].value_counts()

report = f"""# 零售 POS 銷售分析報告

## 資料概要
- 分析筆數：{len(df)} 筆交易
- 總銷售額：${total_rev:,.2f}
- 資料來源：UCI Online Retail II

## 商品銷售 Top 5
{chr(10).join(f'- {r["description"]}: ${r["total_revenue"]:,.2f}' for _, r in top_products.iterrows())}

## 各國銷售 Top 5
{chr(10).join(f'- {r["country"]}: ${r["revenue"]:,.2f} ({r["customers"]} 客戶)' for _, r in top_countries.iterrows())}

## 品類分佈（LLM 分析 {len(df_analyzed)} 筆）
{chr(10).join(f'- {cat}: {cnt} 筆' for cat, cnt in cat_dist.items())}

## 建議
1. 季節商品佔比高 → 提前備貨
2. United Kingdom 佔絕大多數營收 → 其他市場有成長空間
3. 定期追蹤 Top 客戶消費趨勢

## Pipeline
CSV → pandas → SQLite(raw/cleaned/analyzed) → SQL → LLM → 本報告
"""
os.makedirs("output", exist_ok=True)
with open("output/report.md", "w") as f:
    f.write(report)
print("✅ 報告已存到 output/report.md")


---
## Section 7：打包確認


In [ ]:
checks = [("pipeline.db", "SQLite"), ("processed/product_stats.csv", "商品統計"),
          ("processed/country_stats.csv", "國家統計"), ("output/report.md", "報告")]
all_ok = True
for path, desc in checks:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {desc}: {path}")
    if not exists: all_ok = False
if os.path.exists("pipeline.db"):
    c = sqlite3.connect("pipeline.db")
    for t in ["raw_orders", "cleaned_orders", "analyzed_orders"]:
        try:
            n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", c)["n"][0]
            print(f"  ✅ {t}: {n} 筆")
        except: print(f"  ❌ {t}"); all_ok = False
    c.close()
print("\n🎉 全部完成！" if all_ok else "\n⚠️ 有缺漏")


---
## Section 8：FastAPI


In [ ]:
!pip install -q fastapi uvicorn nest_asyncio
from fastapi import FastAPI
import nest_asyncio
nest_asyncio.apply()

api = FastAPI(title="零售銷售分析 API")

@api.get("/health")
def health():
    return {"status": "ok"}

@api.get("/stats")
def get_stats():
    c = sqlite3.connect("pipeline.db")
    df = pd.read_sql("SELECT description, COUNT(*) as orders, ROUND(SUM(total_amount),2) as revenue FROM cleaned_orders GROUP BY description ORDER BY revenue DESC LIMIT 20", c)
    c.close()
    return df.to_dict(orient="records")

@api.get("/analyzed")
def get_analyzed():
    c = sqlite3.connect("pipeline.db")
    df = pd.read_sql("SELECT description, category, llm_insight FROM analyzed_orders LIMIT 20", c)
    c.close()
    return df.to_dict(orient="records")

print("✅ API 定義完成")


In [ ]:
import threading, uvicorn, time, requests as req
thread = threading.Thread(target=uvicorn.run, args=(api,), kwargs={"host":"0.0.0.0","port":8000,"log_level":"warning"})
thread.daemon = True
thread.start()
time.sleep(2)
print("📡 /health:", req.get("http://localhost:8000/health").json())
print("📡 /stats:", req.get("http://localhost:8000/stats").json()[:3])
print("📡 /analyzed:", req.get("http://localhost:8000/analyzed").json()[:3])


> 完整版在 `api.py`。本地：`uvicorn api:app --reload --port 8000`


---
## Section 9：Dashboard


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

df_dash = pd.read_sql("SELECT * FROM cleaned_orders", conn)
country_dropdown = widgets.Dropdown(
    options=["全部"] + sorted(df_dash["country"].unique().tolist()),
    description="選國家："
)

def update_dashboard(country):
    clear_output(wait=True)
    display(country_dropdown)
    data = df_dash if country == "全部" else df_dash[df_dash["country"] == country]
    print(f"📊 {country}: {len(data)} 筆, 營收 ${data['total_amount'].sum():,.2f}")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    data.groupby("description")["total_amount"].sum().sort_values().tail(10).plot.barh(ax=axes[0], color="steelblue")
    axes[0].set_title(f"商品銷售額 Top 10 — {country}")
    if "hour" in data.columns:
        data.groupby("hour")["total_amount"].sum().plot(ax=axes[1], color="coral", marker="o")
        axes[1].set_title(f"每小時銷售額 — {country}")
    plt.tight_layout()
    plt.show()

widgets.interact(update_dashboard, country=country_dropdown)


> 完整版在 `app.py`。本地：`streamlit run app.py`


---
## Section 10：本地部署指引

```bash
cd data/raw/topic_1
uvicorn api:app --reload --port 8000    # Terminal 1
streamlit run app.py                     # Terminal 2
```
